# Writing SMILES with RDKit

This notebook is a first practical step from chemistry into Python and RDKit. It complements the [RDKit documentation](https://www.rdkit.org/docs/).

By the end you should be able to:

- write SMILES for small organic molecules, including branches, rings, aromatic atoms and stereochemistry
- turn a SMILES string into an RDKit molecule and draw it
- inspect the atoms, bonds and atom indices in a molecule
- write a molecule back out as a canonical SMILES
- recognise when RDKit can't read a SMILES
- load a set of molecules from a file

Work from top to bottom. Cells marked **Exercise** are for you to try: edit the SMILES string and run the cell. A hidden solution follows each one - click "Solution" to show it, but have a go first!

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from IPython.display import display

## What is a SMILES string?

**SMILES** (*Simplified Molecular Input Line Entry System*) describes a molecule as a single line of text. Atom symbols are written in order along a path through the molecule, and bond symbols, parentheses and ring digits describe how the atoms are connected.

A SMILES is not a name or a picture: it has no coordinates, so RDKit works out a layout when it draws the molecule. There is usually more than one valid SMILES for the same molecule, because you can start the path at different atoms.

Hydrogens are usually left out, and RDKit adds them using normal valence rules. For example, `CCO` means a carbon bonded to a carbon bonded to an oxygen, which RDKit reads as ethanol.

| Feature | SMILES example | What it represents |
| --- | --- | --- |
| Straight chain | `CCO` | Ethanol: C–C–O |
| Double bond | `C=C` | Ethene |
| Triple bond | `C#N` | A carbon–nitrogen triple bond |
| Branch | `CC(O)C` | Propan-2-ol; O branches from the middle C |
| Carbonyl | `CC=O` | Ethanal; C=O |
| Ring closure | `C1CC1` | Cyclopropane; the two `1` labels join |
| Aromatic atoms | `c1ccccc1` | Benzene; lowercase `c` means aromatic carbon |
| Stereochemistry | `N[C@@H](C)C(=O)O` | One stereoisomer of alanine |

This table is a preview. Each feature is explained and practised below.

## From SMILES to an RDKit molecule

`Chem.MolFromSmiles()` turns a SMILES string into an RDKit molecule. If a molecule is on the last line of a cell, Jupyter draws it automatically:

In [ ]:
ethanol = Chem.MolFromSmiles('CCO')
ethanol

To draw a molecule that isn't on the last line, use `display()`. The exercise cells below use this so they can draw a molecule and then print some information about it.

## 1. Straight chains

To write a straight-chain SMILES: (1) list the non-hydrogen atoms from one end of the chain to the other; (2) write their element symbols in that order; (3) leave out ordinary single-bond symbols and let RDKit infer the hydrogens. For ethanol, C–C–O becomes `CCO`.

`Chem.AddHs()` makes the inferred hydrogens explicit, so we can count them: `GetNumAtoms()` counts all atoms and `GetNumHeavyAtoms()` counts only the non-hydrogen atoms.

### Exercise 1

1-propanamine has three carbons followed by nitrogen (C–C–C–N). Add N to the starter string in the cell below and run it.

In [ ]:
propanamine_smiles = 'CCC'  # Add the terminal nitrogen.
propanamine = Chem.MolFromSmiles(propanamine_smiles)
display(propanamine)

propanamine_with_h = Chem.AddHs(propanamine)
print(f'Propanamine contains: {propanamine_with_h.GetNumAtoms()} atoms, {propanamine_with_h.GetNumHeavyAtoms()} heavy atoms')

<details>
<summary>Solution</summary>

```python
propanamine_smiles = 'CCCN'
```

1-propanamine has 13 atoms in total: four heavy atoms and nine hydrogens.

</details>

## 2. Double bonds

Leaving out a bond symbol means a single bond. Write `=` **between** two atoms to make their bond double. The table's `C=C` is ethene. Notice that each carbon needs fewer inferred hydrogens than it would in ethane, `CC`.

### Exercise 2

Propene is a three-carbon chain with one terminal C=C bond. Change the starter `CCC` into propene. You can write the chain in either direction.

In [ ]:
propene_smiles = 'CCC'  # Add one double bond.
propene = Chem.MolFromSmiles(propene_smiles)
display(propene)

<details>
<summary>Solution</summary>

```python
propene_smiles = 'C=CC'   # 'CC=C' is also correct
```

</details>

## 3. Triple bonds

A triple bond uses `#` between atoms. The short example `C#N` shows the symbol's position, but a real nitrile often has a carbon group attached to the nitrile carbon.

### Exercise 3

Acetonitrile is CH3–C≡N. The starter below has the right atoms; change only the C–N bond order.

In [ ]:
acetonitrile_smiles = 'CCN'  # Change the last bond to a triple bond.
acetonitrile = Chem.MolFromSmiles(acetonitrile_smiles)
display(acetonitrile)

<details>
<summary>Solution</summary>

```python
acetonitrile_smiles = 'CC#N'
```

</details>

## 4. Branches

SMILES follows one path through a molecule. To attach something off that path, write the branch in parentheses immediately after the atom it attaches to. In `CC(O)C`, the `(O)` is attached to the middle carbon. That gives propan-2-ol rather than the straight-chain isomer `CCCO` (propan-1-ol).

### Exercise 4

2-methylbutane has a four-carbon path with a methyl branch on carbon 2. Start with `CCCC`, then add a `(C)` branch at the correct place.

In [ ]:
methylbutane_smiles = 'CCCC'  # Add a methyl branch on carbon 2.
methylbutane = Chem.MolFromSmiles(methylbutane_smiles)
display(methylbutane)

<details>
<summary>Solution</summary>

```python
methylbutane_smiles = 'CC(C)CC'
```

</details>

## 5. Carbonyls

A **carbonyl** has a carbon double-bonded to oxygen: `C=O`. Ethanal can be written `CC=O`. A carbonyl carbon can also have a branch: in propanone, the carbonyl carbon sits between two methyl groups, so the oxygen branches off that middle carbon as `(=O)`.

### Exercise 5

Add a carbonyl oxygen to the middle carbon of the starter `CCC` to make propanone (acetone).

In [ ]:
propanone_smiles = 'CCC'  # Add (=O) to the middle carbon.
propanone = Chem.MolFromSmiles(propanone_smiles)
display(propanone)

<details>
<summary>Solution</summary>

```python
propanone_smiles = 'CC(=O)C'
```

</details>

## 6. Rings

A ring needs one extra connection back to an atom you have already visited. Place the same digit on the two atoms to be joined. The digit is a ring-closure label, not an atom or an atom number. In `C1CC1`, the first and last carbon labelled `1` are connected, making cyclopropane.

### Exercise 6

Cyclohexane has six carbon atoms in a ring. Add matching ring labels to the ends of the six-carbon starter chain.

In [ ]:
cyclohexane_smiles = 'CCCCCC'  # Connect the two ends of the chain.
cyclohexane = Chem.MolFromSmiles(cyclohexane_smiles)
display(cyclohexane)

<details>
<summary>Solution</summary>

```python
cyclohexane_smiles = 'C1CCCCC1'
```

</details>

## 7. Aromatic atoms

Aromatic atoms are written in lowercase. Benzene is `c1ccccc1`: the six `c` atoms are aromatic carbons, and the matching `1` labels close the ring. Pyridine has the same aromatic six-membered ring, but one carbon is replaced by aromatic nitrogen, written `n`.

You can also write benzene with alternating double bonds, `C1=CC=CC=C1`. Section 9 shows that RDKit treats both versions as the same molecule.

### Exercise 7

Change one `c` in benzene's starter SMILES to `n` to make pyridine. Keep all six ring atoms and the ring closure.

In [ ]:
pyridine_smiles = 'c1ccccc1'  # Replace one aromatic carbon with n.
pyridine = Chem.MolFromSmiles(pyridine_smiles)
display(pyridine)

<details>
<summary>Solution</summary>

```python
pyridine_smiles = 'n1ccccc1'
```

It doesn't matter which `c` you change: all six positions in the ring are equivalent.

</details>

## 8. Stereochemistry

Some molecules have the same atoms and bonds but different three-dimensional arrangements. SMILES can record tetrahedral stereochemistry with `@` or `@@` on a bracketed atom. In `N[C@@H](C)C(=O)O`, `[C@@H]` describes the central carbon of **S-alanine**. The `@@` symbol depends on the order in which the neighbouring atoms are written; it is not a universal label for S.

`Chem.FindMolChiralCenters()` asks RDKit to assign R or S to each stereocentre. It gives the atom number and the label, or `?` if the stereochemistry isn't specified.

### Exercise 8

The starter `NC(C)C(=O)O` has no specified stereochemistry. Keep the same atom order and use the opposite marker, `[C@H]`, to specify **R-alanine**.

In [ ]:
r_alanine_smiles = 'NC(C)C(=O)O'  # Specify the central stereocentre.
r_alanine = Chem.MolFromSmiles(r_alanine_smiles)
display(r_alanine)
print('Assigned stereocentres:', Chem.FindMolChiralCenters(r_alanine, includeUnassigned=True))

<details>
<summary>Solution</summary>

```python
r_alanine_smiles = 'N[C@H](C)C(=O)O'
```

RDKit should print `[(1, 'R')]`: atom 1 (the central carbon) is an R stereocentre.

</details>

## 9. Writing SMILES: canonical SMILES

`Chem.MolToSmiles()` does the reverse of `MolFromSmiles`: it writes a molecule out as a SMILES string. With the same RDKit version and writing settings, equivalent representations of the same recorded structure give the same output, regardless of which atom the original SMILES started from. This is called the **canonical SMILES**, and it is a simple way to check whether two SMILES describe the same molecule:

In [ ]:
print(Chem.MolToSmiles(Chem.MolFromSmiles('CC(O)C')))
print(Chem.MolToSmiles(Chem.MolFromSmiles('OC(C)C')))

Both give `CC(C)O`, so they are the same molecule (propan-2-ol).

When RDKit reads a SMILES, it also works out which rings are aromatic and stores those atoms as aromatic, even if the SMILES was written with alternating double bonds:

In [ ]:
Chem.MolToSmiles(Chem.MolFromSmiles('C1=CC=CC=C1'))

You'll see why this matters when we search for substructures in the next notebook.

### Exercise 9

Toluene is benzene with a methyl group, `Cc1ccccc1`. Write toluene a different way in `toluene_2`, for example starting from a ring atom or using alternating double bonds, and check that both give the same canonical SMILES.

In [ ]:
toluene_1 = 'Cc1ccccc1'
toluene_2 = 'Cc1ccccc1'  # Write toluene a different way.

print(Chem.MolToSmiles(Chem.MolFromSmiles(toluene_1)))
print(Chem.MolToSmiles(Chem.MolFromSmiles(toluene_2)))

<details>
<summary>Solution</summary>

Any of these work, and all give `Cc1ccccc1`:

```python
toluene_2 = 'c1ccccc1C'
```

```python
toluene_2 = 'CC1=CC=CC=C1'
```

</details>

### A limit of canonical SMILES

Canonical SMILES gives a consistent way to write the same recorded structure using the same RDKit version and settings. It does not automatically turn a salt into its free base, change a protonation state, or choose one tautomer for you. Those forms can remain different structures with different canonical strings. Notebook 3 will show why this matters when comparing properties.

## 10. Inspecting atoms and bonds

An RDKit molecule contains atom objects and bond objects. We can ask about them directly, rather than only drawing the molecule or counting atoms.

`GetAtoms()` gives the atoms in the molecule. The loop below prints each atom's **index** with `GetIdx()` and its element with `GetSymbol()`. Indices start at 0. In ethanol made from `CCO`, the oxygen is atom 2. Its hydrogens are implicit here, so they do not appear as separate atom objects.

In [ ]:
for atom in ethanol.GetAtoms():
    print(atom.GetIdx(), atom.GetSymbol())

The drawing below labels those same indices. They identify atoms in this particular RDKit object; they are not ring digits or permanent chemical atom numbers. Reading a different atom order, such as `OCC`, can give the same molecule different indices. Substructure searches in notebook 2 return these indices.

In [ ]:
IPythonConsole.drawOptions.addAtomIndices = True
display(ethanol)
IPythonConsole.drawOptions.addAtomIndices = False

`GetBonds()` gives the bonds between explicit atoms. Each bond has two endpoint indices and a bond type. Ethanol has a C–C single bond and a C–O single bond; its implicit hydrogen bonds are not listed.

In [ ]:
for bond in ethanol.GetBonds():
    print(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx(), bond.GetBondType())

To retrieve one atom directly, use `GetAtomWithIdx(index)`. To count the explicit bonds, use `GetNumBonds()`.

### Exercise 10

Print the element symbol of ethanol's atom 2, then print the number of bonds in the molecule.

In [ ]:
# Exercise 10: your code here

<details>
<summary>Solution</summary>

```python
print(ethanol.GetAtomWithIdx(2).GetSymbol())
print(ethanol.GetNumBonds())
```

You should see `O` and `2`. We are inspecting the molecule before adding explicit hydrogens.
</details>

## 11. When RDKit can't read a SMILES

If a SMILES has a mistake, `Chem.MolFromSmiles()` prints an error message and gives back `None` instead of a molecule. For example, this ring is never closed:

In [ ]:
broken = Chem.MolFromSmiles('C1CC')
print(broken)

It's good practice to check for this with `if mol is not None`, especially when reading lots of molecules from a file. The later notebooks do this to skip any entries that can't be read.

### Exercise 11

For each SMILES below, predict whether RDKit can read it **before** running the cell. Then run it to check, and read the error messages for the ones that fail.

In [ ]:
test_smiles = ['CC(C', 'O=C=O', 'c1cccc1', 'CN(C)(C)C', 'C[N+](C)(C)C']

for smiles in test_smiles:
    mol = Chem.MolFromSmiles(smiles)
    print(smiles, 'can be read:', mol is not None)

<details>
<summary>Solution</summary>

- `CC(C` fails: the branch is never closed.
- `O=C=O` (carbon dioxide) is fine.
- `c1cccc1` fails: RDKit can't make a five-carbon ring aromatic ("Can't kekulize mol").
- `CN(C)(C)C` fails: a neutral nitrogen can't have four bonds ("Explicit valence ... is greater than permitted").
- `C[N+](C)(C)C` is fine: writing the nitrogen in square brackets with a `+` charge allows four bonds (a tetramethylammonium ion).

</details>

## 12. Loading SMILES from a file

SMILES often come as a **column in a file** rather than being typed one at a time. The small file `../data/smiles_practice.csv` contains molecules from this lesson. (The `../` means "go up one folder": from `notebooks` to the main folder, where `data` is.) The first few lines of the file look like this:

```
name,smiles
ethanol,CCO
propene,C=CC
propan-2-ol,CC(O)C
```

`Chem.SmilesMolSupplier` reads a file like this. We tell it how the file is laid out:

- `delimiter=','` - the columns are separated by commas
- `smilesColumn=1` and `nameColumn=0` - which columns hold the SMILES and the name (counting from 0)
- `titleLine=True` - the first line holds column headings, not a molecule

The supplier gives us one molecule at a time as we loop over it, so we use a list comprehension to put them all in a list, skipping any that can't be read:

In [ ]:
supplier = Chem.SmilesMolSupplier('../data/smiles_practice.csv', delimiter=',',
                                  smilesColumn=1, nameColumn=0, titleLine=True)
molecules = [mol for mol in supplier if mol is not None]

len(molecules)

RDKit stores each molecule's name in a **property** called `_Name`. `GetProp()` reads a property:

In [ ]:
molecules[0].GetProp('_Name')

`Draw.MolsToGridImage` draws a list of molecules in a grid:

In [ ]:
Draw.MolsToGridImage(molecules, molsPerRow=4)

### Exercise 12

Redraw the grid with each molecule's name underneath it. `Draw.MolsToGridImage` takes a `legends` argument: a list of labels, one per molecule.

In [ ]:
# Exercise 12: your code here

<details>
<summary>Solution</summary>

```python
names = [mol.GetProp('_Name') for mol in molecules]
Draw.MolsToGridImage(molecules, legends=names, molsPerRow=4)
```

</details>

You can now write SMILES for common organic features, turn them into RDKit molecules, inspect atoms and bonds, compare molecules using canonical SMILES, and load molecules from a file. The next notebook introduces SMARTS, a related notation for asking substructure questions.